The opus magnum to complete excel sheet task for FAR-OUT

In [ ]:
from rocketpy import Rocket, Environment, Flight, LiquidMotor, Fluid, CylindricalTank, MassFlowRateBasedTank, MassBasedTank
from math import exp
from datetime import datetime, timedelta
from CoolProp.CoolProp import PropsSI
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import setup
import excel_sheet_functions as ex

In [ ]:
env = setup.get_env_reanalysis(date=(2025,5,30,15))

In [ ]:
# Stałe dla turbulencji
external_tank_diameter = 0.2
tank_height = 1.13
thickness_tank = 0.005
thickness_piston = 0.01

# Parametry, które potrzebuję
p_0 = 63e5 #ciśnienie początkowe
piston_position = 0.892
total_oxidizer_mass = 10.5 # setup for 13kg total propellant mass
flux_time = 5 
ethanol_temperature = 300
gas_initial_mass_fuel = 0

# Parametry, które wyliczam
N20_liq_density = PropsSI("D", "P", p_0, "Q", 0, "NitrousOxide")
N20_gas_density = PropsSI("D", "P", p_0, "Q", 1, "NitrousOxide")
ethanol_liq_density = PropsSI("D", "P", p_0-1e5, "T", ethanol_temperature, "Ethanol") # PropSI pokazuje 63e5 powyżej krytycznego, dlatego odejmuję
ethanol_gas_density = PropsSI("D", "P", p_0-1e5, "Q", 1, "Ethanol")

volume_tank = 0.25*np.pi*(external_tank_diameter-2*thickness_tank)**2*tank_height
volume_piston = 0.25*np.pi*(external_tank_diameter-2*thickness_tank)**2*thickness_piston
volume_oxidizer = piston_position*volume_tank
volume_fuel = volume_tank - volume_oxidizer - volume_piston

gas_initial_mass_ox = (volume_oxidizer - (total_oxidizer_mass / N20_liq_density)) / (1/N20_gas_density - 1/N20_liq_density)
liquid_initial_mass_ox = total_oxidizer_mass - gas_initial_mass_ox
liquid_initial_mass_fuel = volume_fuel * ethanol_liq_density

# Parametry stricte pod tank_geometry
tank_radius = (external_tank_diameter - 2*thickness_tank) / 2
adjusted_height_ox = piston_position*tank_height
adjusted_height_fuel = tank_height - adjusted_height_ox - thickness_piston

# Dla sprawdzenia czy wychodzi mi to samo co volume_oxidizer i volume_fuel
fuel_tank_volume = np.pi * tank_radius**2 * adjusted_height_fuel
oxidizer_tank_volume = np.pi * tank_radius**2 * adjusted_height_ox

In [ ]:
oxidizer_liq = Fluid(name="N2O_l", density=N20_liq_density)
oxidizer_gas = Fluid(name="N2O_g", density=N20_gas_density)
fuel_liq = Fluid(name="ethanol_l", density=ethanol_liq_density) 
fuel_gas = Fluid(name="ethanol_g", density=ethanol_gas_density)

In [ ]:
# Tank geometry
oxidizer_tank_geometry = CylindricalTank(
    radius=tank_radius,
    height=adjusted_height_ox+0.01,
)
fuel_tank_geometry = CylindricalTank(
    radius=tank_radius,
    height=adjusted_height_fuel,
)

In [ ]:
# Define tanks
mass_flow_rate_liq = 1.6*(liquid_initial_mass_ox/total_oxidizer_mass)
mass_flow_rate_gas = 1.6*(gas_initial_mass_ox/total_oxidizer_mass)

oxidizer_tank = MassFlowRateBasedTank(
    name="oxidizer tank",
    geometry=oxidizer_tank_geometry,
    flux_time=flux_time+1.56, 
    initial_liquid_mass=liquid_initial_mass_ox, 
    initial_gas_mass=gas_initial_mass_ox,
    liquid_mass_flow_rate_in=0,
    liquid_mass_flow_rate_out=mass_flow_rate_liq, 
    gas_mass_flow_rate_in=0,
    gas_mass_flow_rate_out=mass_flow_rate_gas, 
    liquid=oxidizer_liq,
    gas=oxidizer_gas,
)

In [ ]:
#Fuel tank
fuel_mass_flow_rate = 0.5
fuel_tank = MassFlowRateBasedTank(
    name="fuel tank",
    geometry=fuel_tank_geometry,
    flux_time=flux_time,
    initial_liquid_mass=liquid_initial_mass_fuel-0.00001, #Same as above, only guess
    initial_gas_mass=gas_initial_mass_fuel,
    liquid_mass_flow_rate_in=0,
    liquid_mass_flow_rate_out=fuel_mass_flow_rate, #heuristics
    gas_mass_flow_rate_in=0,
    gas_mass_flow_rate_out=0,
    liquid=fuel_liq,
    gas=fuel_gas,
)

In [ ]:
motor = setup.create_motor(
    tanks=(oxidizer_tank, fuel_tank), 
    thrust_curve_file="data\\AGH-SS_Z4000-13kgload.eng", #"mean_thrust.csv", #data\\AGH-SS_Z4000-13kgload.eng", #"fitted_thrust.csv", #"raw_thrust.csv"
    burn_time=12,
)
motor.all_info()
# expected curve 23884.758 Ns
# fitted thrust 23881.383 Ns

# mean: 28088.522 Ns
# fitted: 25400.488 Ns

# mean 28841.911 Ns
# fitted: 29096.783 Ns


In [ ]:
rocket = setup.create_rocket(motor=motor, mass=55.567, power_on_drag="data\\cd_or_13kg_on.csv", power_off_drag="data\\cd_or_13kg_off.csv")

In [ ]:
flight = setup.create_flight(rocket=rocket, env=env, inclination=90)

In [ ]:
#flight.all_info()
rocket.all_info()

In [34]:
main_event = next(event for event in flight.parachute_events if event[1].name.lower() == "main")
trigger_time = main_event[0]
main_parachute = main_event[1]
opening_time = 61.876
q_at_opening = flight.dynamic_pressure(opening_time)
peak_force = q_at_opening * main_parachute.cd_s
print(f"Dynamic pressure at main chute opening: {q_at_opening:.2f} Pa")
print(f"Czas pełnego otwarcia maina: {opening_time:.2f} s")
print(f"Maksymalna siła otwarcia (Peak Force): {peak_force:.2f} N")

Dynamic pressure at main chute opening: 470.51 Pa
Czas pełnego otwarcia maina: 61.88 s
Maksymalna siła otwarcia (Peak Force): 5984.88 N


In [35]:
a_main = flight.acceleration(61.914)
a_drogue = flight.acceleration(23.162)
print(f"Acceleration at main chute deployment: {a_main:.2f} m/s^2")
print(f"Acceleration at drogue chute deployment: {a_drogue:.2f} m/s^2")

Acceleration at main chute deployment: 0.11 m/s^2
Acceleration at drogue chute deployment: 9.76 m/s^2


In [36]:
twr = ex.average_thrust_during_rail_phase(flight, motor, rocket, t_start=0)
rail_departure_velocity = ex.rail_departure_velocity_in_ft_per_sec(flight)
rail_departure_velocity_m_per_s = flight.out_of_rail_velocity
max_acceleration_on_power = flight.max_acceleration_power_on
max_acceleration_on_power_time = flight.max_acceleration_power_on_time
print(f"TWR at liftoff: {twr:.8f}")
print(f"Rail departure velocity: {rail_departure_velocity:.8f} ft/s")
print(f"Rail departure velocity (m/s): {rail_departure_velocity_m_per_s:.8f} m/s")
print(f"Max acceleration on power: {max_acceleration_on_power:.8f} m/s^2")
print(f"Time of max acceleration on power: {max_acceleration_on_power_time:.8f} s")

TWR at liftoff: 6.27897199
Rail departure velocity: 118.30300605 ft/s
Rail departure velocity (m/s): 36.05875509 m/s
Max acceleration on power: 53.54991733 m/s^2
Time of max acceleration on power: 0.03579989 s


In [ ]:
max_q_time = flight.max_dynamic_pressure_time
max_mach_time = flight.max_mach_number_time
max_q_altitude = flight.altitude(max_q_time)
max_speed = flight.max_speed
max_speed_time = flight.max_speed_time
max_mach_number = flight.max_mach_number
print(f"Time of max dynamic pressure: {max_q_time:.4f} s")
print(f"Time of max Mach number: {max_mach_time:.4f} s")
print(f"Altitude at max dynamic pressure: {max_q_altitude:.4f} m")
print(f"Max speed: {max_speed:.4f} m/s")
print(f"Time of max speed: {max_speed_time:.4f} s")
print(f"Max Mach number: {max_mach_number:.4f}")

In [ ]:
apogee = flight.altitude(flight.apogee_time)
apogee_time = flight.apogee_time
print(f"Apogee: {apogee:.4f} m")
print(f"Time of apogee: {apogee_time:.4f} s")

In [ ]:
cg_position_from_tail = 4.99 - rocket.center_of_mass(0)
cp_position_from_tail = 4.99 - rocket.cp_position(0)
static_stability = rocket.stability_margin(0, 0)
print(f"Maximum static margin during flight in % of body lenght: {flight.max_stability_margin*0.2/4.49*100:.4f} percent of body lenght")
print(f"Minimum static margin during flight in % of body length: {flight.min_stability_margin*0.2/4.49*100:.4f} percent of body length")
print(f"Center of mass position from tail: {cg_position_from_tail:.4f} m")
print(f"Center of pressure position from tail: {cp_position_from_tail:.4f} m")
print(f"Static stability margin: {static_stability:.4f} calibers")


In [ ]:
ex.analyze_advanced_damping(flight, signal_name="partial_angle_of_attack")
ex.analyze_and_plot_damping(flight)

In [37]:
max_distance = ex.distance_from_pad(flight)
print(f"Max distance from pad(nominal): {max_distance:.2f} m")

Max distance from pad(nominal): 424.99 m


In [38]:
#max distance ballistic: no parachutes
rocket_no_parachutes = setup.create_rocket(motor=motor, no_main=True, no_drogue=True)
flight_no_parachutes = setup.create_flight(rocket=rocket_no_parachutes, env=env, inclination=90)
distance_balistic = ex.distance_from_pad(flight_no_parachutes)
print(f"Max distance from pad (ballistic, no parachutes): {distance_balistic:.2f} m")

Max distance from pad (ballistic, no parachutes): 80.26 m


In [39]:
#max distance drougue parachute only
rocket_drogue_only = setup.create_rocket(motor=motor, no_main=True, no_drogue=False)
flight_drogue_only = setup.create_flight(rocket=rocket_drogue_only, env=env, inclination=90)
distance_drogue_only = ex.distance_from_pad(flight_drogue_only)
print(f"Max distance from pad (drogue parachute only): {distance_drogue_only:.2f} m")

Max distance from pad (drogue parachute only): 285.61 m


In [40]:
# max distance main at apogee
rocket_main_only = setup.create_rocket(motor=motor, no_main=True, no_drogue=True, main_at_apogee=True)
flight_main_only = setup.create_flight(rocket=rocket_main_only, env=env, inclination=90)
distance_main_only = ex.distance_from_pad(flight_main_only)
print(f"Max distance from pad (main parachute at apogee): {distance_main_only:.2f} m")

Max distance from pad (main parachute at apogee): 854.28 m


In [ ]:
#max pitch/jaw moment
print(ex.max_yaw_moment(flight))
print(ex.max_pitch_moment(flight))

In [ ]:
ex.plot_aerodynamic_stability(rocket, flight)

In [ ]:
ex.plot_cp_vs_mach_number(rocket, flight)

In [ ]:
t_exit = flight.out_of_rail_time 
g = 9.80665  
t_start = 0
rail_times = np.linspace(t_start, t_exit, 100)

# Calculate Average Thrust 
thrust_values = [motor.thrust(t) for t in rail_times]
avg_thrust = np.mean(thrust_values)

# Calculate Average Mass during rail phase
propellant_mass_values = [motor.propellant_mass(t) for t in rail_times]
avg_propellant_mass = np.mean(propellant_mass_values)
avg_total_mass = rocket.mass + avg_propellant_mass # rocket.mass is dry mass

# Calculate TWR
twr_rail_avg = avg_thrust / (avg_total_mass * g)
print(f"Average Thrust during rail phase: {avg_thrust:.2f} N")
print(f"Average Total Mass during rail phase: {avg_total_mass:.2f} kg")
print(f"TWR at liftoff (average during rail phase): {twr_rail_avg:.2f}")

In [ ]:
rocket.thrust_to_weight.plot()


In [ ]:
damping_ratio_df = pd.read_csv("data\\damping_ratio_3.csv", header=None)

damping_ratio_df.columns = ["time", "damping_ratio"]
damping_ratio_df.dropna(inplace=True)
max_value = damping_ratio_df["damping_ratio"].max()
min_value = damping_ratio_df["damping_ratio"].min()
print(f"Maximum damping ratio: {max_value:.4f}")
print(f"Minimum damping ratio: {min_value:.4f}")
damping_ratio_df.plot(x="time", y="damping_ratio", title="Damping Ratio over Time", xlabel="Time (s)", ylabel="Damping Ratio", grid=True, legend=True)
# cut off at time = 20s for better visualization
damping_ratio_df = damping_ratio_df[damping_ratio_df["time"] <= 20]
time_values = damping_ratio_df["time"]
damping_ratio_values = damping_ratio_df["damping_ratio"]
plt.figure(figsize=(10, 6))
plt.plot(time_values, damping_ratio_values, label="Damping Ratio", color="blue")
plt.title("Damping Ratio over Time")
plt.xlabel("Time (s)")
plt.ylabel("Damping Ratio")
plt.legend()
plt.grid()
plt.show()